# DRCD: official OpenAI API cost frontier (offline-first)
Native cells are the source: unchanged CPU manifest, payload, persistent budget ledger,
metadata counting, HTTP, scoring, dev-only decision, test dispatch and plot.
Default **replay** never reads credentials/ledger or calls HTTP. It regrades saved API results, or reports not_run when no results exist.
Only an explicitly authorized fresh run may send requests. Prices are a 2026-11-10 official-doc snapshot;
model IDs are aliases (no invented dated snapshots), reasoning none is not deterministic decoding.


## 固定資料、計費與驗證邊界
- 唯一資料入口是保留的 `results/drcd-cpu/sample-manifest.json`；核對完整檔案 SHA 後直接使用，**不重新抽樣**。
  原來源為 `MediaTek-Research/TCEval-v2`，revision `93ae26a94572907f6d33f9844abd9d074bcc3723`，DRCD public historical test。
  既有 seed=20261106、先依原段落 SHA 分組的 33 段樣本保留 8 dev / 24 test / 1 warmup；來源 row/context hashes 與 references 仍在 manifest。
  這是公開舊 test 的實驗重劃，不保證模型訓練未見；CPU 結果也已公開。DRCD CC-BY-SA 3.0 歸屬見 `data/DRCD-ATTRIBUTION.md`。
- 唯一模型輸入為相同 CPU SYSTEM、完整原段落與原問題。Gold 只在本地評分端；不縮段落、不改題。
- 先兩模型 warmup，再兩模型各 8 dev；只按 dev EM/F1 與 usage-derived USD 選 0/.25/.5/.75/1 small share。
  允許全 Terra／全 Luna，tie 少 small；decision + SHA durable 保存後，才逐題輪轉三方案各實送 24 test。
- 官方文件已由父代理於 2026-11-10 查核，離線保存於 `/home/wens/outputs/drcd-paid-api/docs/`：
  `pricing.md`、`terra.md`、`luna.md`、`input-tokens.md`、`reasoning.md`、`token-counting.md`、`prompt-caching.md`。
  Notebook 使用快照常數，不聯網更新；只有明確啟動 fresh 時才讀取指定私有憑證。兩模型僅有 alias，沒有捏造 dated snapshot。
- US$1 是全部嘗試原授權；有效固定停止線 **US$0.80**。private ledger 跨 run 保留 settled + unresolved reserves，沒有 reset。
  金額用整數 nanoUSD；exact input count +256 <=8192；reserve 用最貴 cache-write 價及 500 output 上界，再乘 2。
  Cache 設 explicit、30m、無 breakpoint；任何實際非零／缺失 cache 計數都停，留 reserve。timeout／HTTP error 亦不釋放。
- Output usage 含 reasoning／不可見 tokens，reasoning subset 不另外重加。usage-derived cost **不等於 billing reconciliation**。
  本機 kill 不能保證 provider 停止計費；持久未結 reserve 才是預算曝險控制。無重試／充值／工具調用。
- 預設 replay，尚無真實 API raw 時只報 `not_run`、不畫圖、不製造結果。離線 fixtures 只測 ledger／解析／路由。
  真正 fresh 必須在預算攔截測試通過後執行 README 明確命令；離線 fixtures 不代表已發生 token-counting 或模型推論。


In [1]:
import collections, contextlib, copy, datetime, decimal, fcntl, hashlib, json, os, stat, time, unicodedata, uuid
from pathlib import Path

CAP_NANO = 800_000_000
AUTH_SCOPE = 'drcd-openai-usd1'
MODELS = ('gpt-6.1-terra', 'gpt-6-luna')
# nanoUSD per token: ordinary input, output, most expensive (12h) cache write.
RATES = {'gpt-6.1-terra': (1000, 5000, 2500), 'gpt-6-luna': (100, 500, 250)}
MAX_OUTPUT = 500

class APIStop(RuntimeError):
    """Only fixed local codes; never include HTTP bodies or credentials."""

def integer(value, minimum=0):
    if type(value) is not int or value < minimum:
        raise APIStop('invalid_tokens')
    return value

def canonical(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':')).encode()

def sha(value):
    return hashlib.sha256(value).hexdigest()

def utc():
    return datetime.datetime.now(datetime.timezone.utc).isoformat()

def atomic_json(path, value, private=False):
    path = Path(path)
    temp = path.with_name('.' + path.name + '.' + uuid.uuid4().hex)
    fd = os.open(temp, os.O_CREAT | os.O_EXCL | os.O_WRONLY, 0o600 if private else 0o644)
    try:
        with os.fdopen(fd, 'wb') as stream:
            stream.write(canonical(value) + b'\n'); stream.flush(); os.fsync(stream.fileno())
        os.replace(temp, path)
        directory = os.open(path.parent, os.O_RDONLY | os.O_DIRECTORY)
        try: os.fsync(directory)
        finally: os.close(directory)
    finally:
        if temp.exists(): temp.unlink()

def reservation(model, exact_count):
    n = integer(exact_count, 1) + 256
    if n > 8192: raise APIStop('short_context_limit')
    return 2 * (n * RATES[model][2] + MAX_OUTPUT * RATES[model][1])

def usage_cost(model, usage, exact_count):
    if not isinstance(usage, dict): raise APIStop('missing_usage')
    i = integer(usage.get('input_tokens'), 1)
    o = integer(usage.get('output_tokens'))
    total = integer(usage.get('total_tokens'))
    if i > exact_count + 256 or i > 8192 or o > MAX_OUTPUT or total != i + o:
        raise APIStop('usage_out_of_bounds')
    detail = usage.get('input_tokens_details')
    if not isinstance(detail, dict): raise APIStop('ambiguous_cache')
    for name in ('cached_tokens', 'cache_write_tokens'):
        if integer(detail.get(name)) != 0: raise APIStop('unsupported_cache')
    writes = detail.get('cache_write_tokens_details', {})
    if not isinstance(writes, dict) or any(integer(v) != 0 for v in writes.values()):
        raise APIStop('unsupported_cache')
    out = usage.get('output_tokens_details', {})
    if not isinstance(out, dict): raise APIStop('invalid_tokens')
    reasoning = integer(out.get('reasoning_tokens', 0))
    if reasoning > o: raise APIStop('invalid_tokens')
    # Reasoning is a subset of output_tokens, never charged twice.
    return i * RATES[model][0] + o * RATES[model][1]

class BudgetLedger:
    """Exclusive lock for the whole fresh run; crash-safe reserve before HTTP.
    Pending exposure survives restarts. No automatic reset/reconciliation/unhalt.
    """
    def __init__(self, path):
        self.path = Path(path)
        self.lock = None
    def __enter__(self):
        self.path.parent.mkdir(parents=True, exist_ok=True, mode=0o700)
        if self.path.parent.is_symlink(): raise APIStop('ledger_symlink')
        os.chmod(self.path.parent, 0o700)
        lockpath = self.path.with_suffix('.lock')
        fd = os.open(lockpath, os.O_RDWR | os.O_CREAT | os.O_NOFOLLOW, 0o600)
        self.lock = os.fdopen(fd, 'a+')
        try:
            fcntl.flock(fd, fcntl.LOCK_EX | fcntl.LOCK_NB)
            if self.path.exists():
                info = self.path.lstat()
                if not stat.S_ISREG(info.st_mode) or info.st_uid != os.getuid() or stat.S_IMODE(info.st_mode) != 0o600:
                    raise APIStop('ledger_permissions')
                self.data = json.loads(self.path.read_text())
                if (self.data.get('schema') != 1 or self.data.get('auth_scope') != AUTH_SCOPE
                        or self.data.get('cap_nano_usd') != CAP_NANO
                        or self.data.get('authorized_nano_usd') != 1_000_000_000):
                    raise APIStop('ledger_scope_or_cap')
                self.validate()
            else:
                self.data = {'schema': 1, 'auth_scope': AUTH_SCOPE, 'cap_nano_usd': CAP_NANO,
                    'authorized_nano_usd': 1_000_000_000, 'entries': {}, 'counts': {},
                    'metadata_request_count': 0, 'inference_request_count': 0, 'halted': False}
                self.save()
            return self
        except BaseException:
            self.__exit__(None, None, None)
            raise
    def __exit__(self, *args):
        if self.lock is not None:
            self.lock.close(); self.lock = None
    def validate(self):
        integer(self.data['metadata_request_count']); integer(self.data['inference_request_count'])
        for entry in self.data['entries'].values():
            if entry['state'] not in ('pending', 'settled'): raise APIStop('ledger_state')
            integer(entry['reserve_nano_usd'], 1)
            if entry['state'] == 'settled':
                if integer(entry['actual_nano_usd']) > entry['reserve_nano_usd']: raise APIStop('ledger_amount')
        if self.exposure() > CAP_NANO: raise APIStop('ledger_exposure')
    def save(self):
        if self.lock is None: raise APIStop('ledger_not_locked')
        atomic_json(self.path, self.data, private=True)
    def exposure(self):
        return sum(e['actual_nano_usd'] if e['state'] == 'settled' else e['reserve_nano_usd']
                   for e in self.data['entries'].values())
    def ensure_active(self):
        if self.data['halted'] or any(e['state'] == 'pending' for e in self.data['entries'].values()):
            raise APIStop('ledger_halted_or_unresolved')
    def reserve(self, model, exact_count, metadata):
        self.ensure_active()
        if self.data['inference_request_count'] >= 90: raise APIStop('request_count_limit')
        amount = reservation(model, exact_count)
        if self.exposure() + amount > CAP_NANO: raise APIStop('budget_exhausted')
        token = uuid.uuid4().hex
        self.data['entries'][token] = {'state': 'pending', 'reserve_nano_usd': amount,
            'model': model, 'exact_input_tokens': exact_count, 'reserved_at': utc(), **metadata}
        self.data['inference_request_count'] += 1
        self.save()  # atomic + fsync must succeed BEFORE transport is invoked
        return token
    def settle(self, token, usage):
        entry = self.data['entries'][token]
        if entry['state'] != 'pending': raise APIStop('already_settled')
        try:
            actual = usage_cost(entry['model'], usage, entry['exact_input_tokens'])
            if actual > entry['reserve_nano_usd']: raise APIStop('reserve_exceeded')
        except APIStop:
            self.halt('ambiguous_usage')
            raise
        entry.update(state='settled', actual_nano_usd=actual, settled_at=utc(), usage=usage)
        self.save()
        return actual
    def halt(self, code):
        self.data['halted'] = True
        self.data['halt_code'] = code
        self.save()
    def snapshot(self):
        return copy.deepcopy(self.data)


In [2]:
def input_count(ledger, sender, payload):
    ledger.ensure_active()
    counted_payload = {'model': payload['model'], 'input': payload['input']}
    key = sha(canonical(counted_payload))
    if key in ledger.data['counts']:
        n = integer(ledger.data['counts'][key], 1)
        reservation(payload['model'], n)
        return n
    ledger.data['metadata_request_count'] += 1
    ledger.save()
    try:
        reply = sender('/responses/input_tokens', counted_payload)
        if reply.get('object') != 'response.input_tokens': raise APIStop('counter_schema')
        n = integer(reply.get('input_tokens'), 1)
        reservation(payload['model'], n)
        ledger.data['counts'][key] = n
        ledger.save()
        return n
    except BaseException:
        ledger.halt('counter_failed')
        raise APIStop('counter_failed') from None

def parsed_response(reply, expected_model):
    if not isinstance(reply, dict) or reply.get('service_tier') != 'default':
        raise APIStop('unexpected_service_tier')
    if reply.get('model') != expected_model: raise APIStop('unexpected_model_alias')
    if reply.get('status') not in ('completed', 'incomplete') or not isinstance(reply.get('id'), str):
        raise APIStop('response_schema')
    texts = []
    for item in reply.get('output', []):
        if item.get('type') == 'reasoning': continue
        if item.get('type') != 'message': raise APIStop('unexpected_output_type')
        for part in item.get('content', []):
            if part.get('type') == 'output_text' and isinstance(part.get('text'), str): texts.append(part['text'])
            elif part.get('type') == 'refusal': continue
            else: raise APIStop('unexpected_content_type')
    answer = ''.join(texts)
    return answer, None if reply['status'] == 'completed' and answer else 'incomplete_or_empty'

def paid_request(ledger, sender, payload, exact_count, metadata, on_reserved=None):
    token = ledger.reserve(payload['model'], exact_count, metadata)
    start = time.monotonic()
    entry = ledger.data['entries'][token]
    record = {**metadata, 'model': payload['model'], 'payload': payload, 'ts': utc(),
        'ledger_entry': token, 'started_monotonic': start, 'exact_input_tokens': exact_count,
        'reserve_nano_usd': entry['reserve_nano_usd'], 'cost_nano_usd': None,
        'unknown_reserve_nano_usd': entry['reserve_nano_usd'], 'answer': '', 'error': None,
        'usage': None, 'response_id': None, 'raw_response': None}
    try:
        if on_reserved is not None: on_reserved(token, record)
        reply = sender('/responses', payload)
        # Keep the safe response even if its usage/cache is unsupported.
        record.update(raw_response=reply, usage=reply.get('usage'), response_id=reply.get('id'))
        answer, error = parsed_response(reply, payload['model'])
        actual = ledger.settle(token, reply.get('usage'))
        record.update(answer=answer, error=error, cost_nano_usd=actual, unknown_reserve_nano_usd=0)
    except BaseException as exc:
        ledger.halt('inference_failed_or_ambiguous')
        record['error'] = 'inference_failed_or_ambiguous'
        record['error_type'] = 'timeout' if isinstance(exc, TimeoutError) else 'request_or_usage_error'
        # HTTP exceptions from our transport carry only a numeric status, never a body/message.
        record['http_status'] = getattr(exc, 'http_status', None)
        failure = APIStop('inference_failed_or_ambiguous')
        failure.record = record
        raise failure from None
    finally:
        record['duration_seconds'] = time.monotonic() - start
        record['metadata_request_count'] = ledger.data['metadata_request_count']
        record['inference_request_count'] = ledger.data['inference_request_count']
        record['budget_exposure_nano_usd'] = ledger.exposure()
    return record


In [3]:
MANIFEST_SHA = '1821c4f1904a51b26a3b87baa2831140064b06f118307bc9882f469fb36d1cf3'
SYSTEM = '請只根據提供的段落回答問題。僅輸出最短的答案原文，不要解釋或加入前綴。'
GRID = (0, .25, .5, .75, 1)
SCHEMES = ('all_terra', 'all_luna', 'frozen_mix')

def load_manifest(path):
    raw = Path(path).read_bytes()
    if sha(raw) != MANIFEST_SHA: raise APIStop('manifest_sha_mismatch')
    manifest = json.loads(raw)
    if [len(manifest[k]) for k in ('dev', 'test', 'warmup')] != [8, 24, 1]: raise APIStop('manifest_splits')
    rows = manifest['dev'] + manifest['test'] + manifest['warmup']
    if len({r['id'] for r in rows}) != 33 or len({r['context_sha256'] for r in rows}) != 33:
        raise APIStop('manifest_context_overlap')
    if any(sha(r['paragraph'].encode()) != r['context_sha256'] for r in rows):
        raise APIStop('manifest_context_sha')
    return manifest, raw

def payload_for(row, model):
    if model not in MODELS: raise APIStop('model_not_allowed')
    return {'model': model, 'input': [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': '段落：' + row['paragraph'] + '\n問題：' + row['question']}],
        'reasoning': {'effort': 'none'}, 'max_output_tokens': MAX_OUTPUT,
        'store': False, 'tools': [], 'service_tier': 'default', 'stream': False,
        'prompt_cache_options': {'mode': 'explicit', 'ttl': '30m'}}

def small_ids(rows, share):
    if share not in GRID: raise APIStop('invalid_share')
    ordered = sorted((r['id'] for r in rows), key=lambda i: sha(('route-v1:' + str(i)).encode()))
    if not float(len(rows) * share).is_integer(): raise APIStop('non_integral_routes')
    return set(ordered[:int(len(rows) * share)])

def normalize(text):
    return ''.join(c for c in unicodedata.normalize('NFKC', text).casefold()
                   if not c.isspace() and not unicodedata.category(c).startswith('P'))

def score(prediction, golds, error=None):
    p = normalize(prediction or '')
    if error or not p or not golds: return 0.0, 0.0
    em, f1 = 0.0, 0.0
    for gold in golds:
        g = normalize(gold)
        em = max(em, float(p == g))
        common = sum((collections.Counter(p) & collections.Counter(g)).values())
        f1 = max(f1, 2 * common / (len(p) + len(g)) if g else 0.0)
    return em, f1

def aggregate(records, denominator):
    return {'denominator': denominator, 'collected': len(records),
        'em': sum(r['em'] for r in records) / denominator,
        'f1': sum(r['f1'] for r in records) / denominator,
        'usage_derived_nano_usd': sum(r.get('cost_nano_usd') or 0 for r in records),
        'unknown_reserve_nano_usd': sum(r.get('unknown_reserve_nano_usd', 0) for r in records),
        'duration_seconds': sum(r['duration_seconds'] for r in records),
        'failed_collected': sum(bool(r['error']) for r in records)}


In [4]:
def select_decision(manifest, records):
    dev = [r for r in records if r['phase'] == 'dev']
    table = {(r['id'], r['model']): r for r in dev}
    if len(dev) != 16 or len(table) != 16: raise APIStop('dev_incomplete')
    baseline = aggregate([table[(row['id'], MODELS[0])] for row in manifest['dev']], 8)
    grid = []
    for share in GRID:
        selected = small_ids(manifest['dev'], share)
        chosen = [table[(r['id'], MODELS[int(r['id'] in selected)])] for r in manifest['dev']]
        if any(r['cost_nano_usd'] is None for r in chosen): raise APIStop('dev_cost_unknown')
        metrics = aggregate(chosen, 8)
        grid.append({'small_share': share, **metrics,
            'eligible': metrics['em'] >= baseline['em'] and metrics['f1'] >= baseline['f1']})
    winner = min((g for g in grid if g['eligible']), key=lambda g: (g['usage_derived_nano_usd'], g['small_share']))
    share = winner['small_share']; selected = small_ids(manifest['test'], share)
    return {'small_share': share, 'label': 'all_luna' if share == 1 else 'all_terra' if share == 0 else 'mixed',
        'objective': 'minimum dev usage-derived USD subject to both EM/F1 >= all Terra; tie fewer Luna',
        'grid': grid, 'test_routes': {r['id']: MODELS[int(r['id'] in selected)] for r in manifest['test']},
        'manifest_sha256': MANIFEST_SHA, 'dev_records_sha256': sha(canonical(dev)),
        'frozen_at': utc(), 'frozen_at_monotonic': time.monotonic(), 'first_test_order': 18}

def initial_plan(manifest):
    plan = [{'id': manifest['warmup'][0]['id'], 'phase': 'warmup', 'scheme': 'warmup', 'model': m} for m in MODELS]
    for index, row in enumerate(manifest['dev']):
        for model in (MODELS if index % 2 == 0 else MODELS[::-1]):
            plan.append({'id': row['id'], 'phase': 'dev', 'scheme': 'dev', 'model': model})
    return plan

def test_plan(manifest, decision):
    plan = []
    for index, row in enumerate(manifest['test']):
        rotated = SCHEMES[index % 3:] + SCHEMES[:index % 3]
        for scheme in rotated:
            model = MODELS[0] if scheme == 'all_terra' else MODELS[1] if scheme == 'all_luna' else decision['test_routes'][row['id']]
            plan.append({'id': row['id'], 'phase': 'test', 'scheme': scheme, 'model': model})
    return plan

def audit_records(manifest, records, decision=None):
    expected = initial_plan(manifest)
    if decision is not None:
        calculated = select_decision(manifest, records)
        for name in ('small_share', 'label', 'objective', 'grid', 'test_routes', 'manifest_sha256', 'dev_records_sha256', 'first_test_order'):
            if calculated[name] != decision[name]: raise APIStop('decision_audit_' + name)
        expected += test_plan(manifest, decision)
    if len(records) > len(expected): raise APIStop('too_many_records_or_unfrozen_test')
    by_id = {r['id']: r for k in ('warmup', 'dev', 'test') for r in manifest[k]}
    for index, (record, task) in enumerate(zip(records, expected)):
        if record['order'] != index or any(record[k] != task[k] for k in ('id', 'phase', 'scheme', 'model')):
            raise APIStop('request_route_audit')
        if record['payload'] != payload_for(by_id[record['id']], record['model']): raise APIStop('payload_audit')
        em, f1 = score(record['answer'], by_id[record['id']]['references'], record['error'])
        if (em, f1) != (record['em'], record['f1']): raise APIStop('score_audit')
        if record['phase'] == 'test' and record['started_monotonic'] < decision['frozen_at_monotonic']:
            raise APIStop('test_before_freeze')
        if record['cost_nano_usd'] is not None:
            if usage_cost(record['model'], record['usage'], record['exact_input_tokens']) != record['cost_nano_usd']:
                raise APIStop('cost_audit')
        elif not record['error']: raise APIStop('unknown_cost_without_failure')
    return True

def summarize(records, status):
    return {'status': status, 'cost_basis': 'usage-derived standard input/output; NOT billing reconciled',
        'inference_request_count': len(records), 'planned_inference_requests': 90,
        'usage_derived_nano_usd': sum(r.get('cost_nano_usd') or 0 for r in records),
        'unknown_reserve_nano_usd': sum(r.get('unknown_reserve_nano_usd', 0) for r in records),
        'warmup': aggregate([r for r in records if r['phase'] == 'warmup'], 2),
        'dev': {m: aggregate([r for r in records if r['phase'] == 'dev' and r['model'] == m], 8) for m in MODELS},
        'test': {s: aggregate([r for r in records if r['phase'] == 'test' and r['scheme'] == s], 24) for s in SCHEMES}}

def replay_results(out, manifest_path):
    manifest, manifest_bytes = load_manifest(manifest_path)
    raw = Path(out) / 'raw-generations.jsonl'
    if not raw.exists(): return summarize([], 'not_run')
    if (Path(out) / 'sample-manifest.json').read_bytes() != manifest_bytes: raise APIStop('copied_manifest_mismatch')
    records = [json.loads(line) for line in raw.read_text().splitlines() if line]
    inflight_path = Path(out) / 'inflight.json'
    if inflight_path.exists():
        interrupted = json.loads(inflight_path.read_text())
        if interrupted['ledger_entry'] not in {r.get('ledger_entry') for r in records}:
            interrupted.update(error='interrupted_unknown_reserve', em=0.0, f1=0.0,
                answer='', cost_nano_usd=None, duration_seconds=0)
            records.append(interrupted)
    decision = None
    if (Path(out) / 'decision.json').exists():
        data = (Path(out) / 'decision.json').read_bytes()
        proof = json.loads((Path(out) / 'decision-sha256.json').read_text())
        if sha(data) != proof['sha256']: raise APIStop('decision_sha_mismatch')
        decision = json.loads(data)
    audit_records(manifest, records, decision)
    complete = len(records) == 90 and all(r.get('cost_nano_usd') is not None and not r.get('unknown_reserve_nano_usd') for r in records)
    report = summarize(records, 'complete' if complete else 'partial')
    report['decision'] = decision
    report['raw_sha256'] = sha(raw.read_bytes())
    # Read only public budget snapshot, never private ledger or key.
    if (Path(out) / 'budget-snapshot.json').exists():
        report['budget_snapshot'] = json.loads((Path(out) / 'budget-snapshot.json').read_text())
    return report


In [5]:
import urllib.request, urllib.error

API_ORIGIN = 'https://api.openai.com/v1'

def authorize_fresh(mode):
    if mode != 'fresh' or os.environ.get('DRCD_API_MAX_USD') != '0.80':
        raise APIStop('explicit_fresh_0_80_authorization_required')
    path = os.environ.get('DRCD_API_KEY_FILE')
    if not path or not Path(path).is_absolute(): raise APIStop('absolute_key_file_required')
    return Path(path)  # No read, stat, or secret access in authorization/replay.

def safe_reply(reply, secret):
    # Whitelist public response body fields. No response/request headers or error messages.
    allowed = {k: reply[k] for k in ('id', 'model', 'status', 'service_tier', 'usage', 'output', 'created_at', 'incomplete_details') if k in reply}
    def redact(value):
        if isinstance(value, str): return value.replace(secret, '[REDACTED]')
        if isinstance(value, list): return [redact(v) for v in value]
        if isinstance(value, dict): return {str(k).replace(secret, '[REDACTED]'): redact(v) for k, v in value.items()}
        return value
    return redact(allowed)

def make_transport(mode):
    path = authorize_fresh(mode)
    # Only fresh calls this function. Never use OAuth, environment token values or other providers.
    secret = None
    try:
        fd = os.open(path, os.O_RDONLY | os.O_NOFOLLOW | os.O_NONBLOCK)
        with os.fdopen(fd, 'rb') as stream:
            info = os.fstat(stream.fileno())
            if not stat.S_ISREG(info.st_mode) or info.st_uid != os.getuid() or stat.S_IMODE(info.st_mode) != 0o600:
                raise APIStop('key_file_permissions')
            if info.st_size > 4096: raise APIStop('key_file_size')
            secret = stream.read(4097).decode('ascii').strip()
        if not secret or any(c.isspace() for c in secret): raise APIStop('key_file_format')
    except BaseException:
        raise APIStop('key_file_rejected') from None
    class NoRedirect(urllib.request.HTTPRedirectHandler):
        def redirect_request(self, *args, **kwargs): raise APIStop('redirect_forbidden')
    # Disable environment proxy discovery, redirects, SDK retries and provider-base overrides.
    opener = urllib.request.build_opener(urllib.request.ProxyHandler({}), NoRedirect())
    def sender(endpoint, payload):
        if endpoint not in ('/responses', '/responses/input_tokens'): raise APIStop('endpoint_forbidden')
        try:
            request = urllib.request.Request(API_ORIGIN + endpoint, data=canonical(payload), method='POST',
                headers={'Authorization': 'Bearer ' + secret, 'Content-Type': 'application/json'})
            with opener.open(request, timeout=115) as reply:
                if reply.status != 200: raise APIStop('unexpected_http_status')
                data = reply.read(2_000_001)
                if len(data) > 2_000_000: raise APIStop('response_too_large')
                body = json.loads(data)
                if not isinstance(body, dict): raise APIStop('response_not_object')
            if endpoint == '/responses/input_tokens':
                return {'object': body.get('object'), 'input_tokens': body.get('input_tokens')}
            return safe_reply(body, secret)
        except urllib.error.HTTPError as exc:
            status = exc.code
            exc.close()  # Never read body: providers may echo secrets in messages.
            failure = APIStop('http_error'); failure.http_status = status
            raise failure from None
        except TimeoutError:
            raise TimeoutError('request_timeout') from None
        except BaseException:
            raise APIStop('transport_failed') from None
    return sender


In [6]:
# Infrastructure only: fixed Python source, no shell, no model-generated commands.
import os, sys, json, time, subprocess, signal
from pathlib import Path

WATCHDOG_SOURCE = r'''
import os, sys, json, time, signal
from pathlib import Path
out = Path(sys.argv[1]); pid = int(sys.argv[2]); identity = sys.argv[3]
def stat():
    try: return Path(f'/proc/{pid}/stat').read_text().split(') ')[1].split()
    except FileNotFoundError: return None
def write(name, obj):
    p = out / name; t = p.with_suffix('.watchdog.tmp')
    with t.open('w') as f:
        json.dump(obj, f, indent=2); f.flush(); os.fsync(f.fileno())
    t.replace(p)
s = stat()
if not s or s[19] != identity or int(s[2]) != pid or os.getpgrp() == pid:
    sys.exit(91)
write('watchdog-ready.json', {'pid': os.getpid(), 'target_pid': pid, 'target_pgid': pid})
while True:
    reason = None
    s = stat()
    if not s or s[0] == 'Z': reason = 'parentdeath'
    elif s[19] != identity: sys.exit(92)  # never signal a reused PID
    try:
        state = json.loads((out / 'watchdog-control.json').read_text())
        now = time.monotonic()
        for key in ['overall', 'inference', 'request']:
            if state.get(key) is not None and now >= state[key]:
                reason = key + ' hard deadline'; break
        if state.get('abort'): reason = 'exception: ' + state['abort']
        if state.get('done') and not reason:
            write('watchdog.json', {'status':'complete', 'target_pid':pid, 'elapsed_seconds':now-state['started']})
            sys.exit(0)
    except Exception as exc:
        state = {}; reason = 'watchdog control failure: ' + repr(exc)
    if reason:
        evidence = {'status':'partial', 'reason':reason, 'target_pid':pid,
                    'target_pgid':pid, 'request':state.get('active_request'), 'monotonic':time.monotonic()}
        try:
            raw = out / 'raw-generations.jsonl'
            evidence['requests_saved'] = len(raw.read_text().splitlines()) if raw.exists() else 0
            write('failure.json', evidence); write('watchdog.json', evidence)
        finally:
            # Disk/evidence errors must never prevent the deadline's kill.
            # Only the group verified above as this dedicated worker/kernel's own group.
            try: os.killpg(pid, signal.SIGKILL)
            except ProcessLookupError: pass
        sys.exit(0)
    time.sleep(.02)
'''

class ResourceGuard:
    def __init__(self, out, overall_seconds=1200):
        self.pid = os.getpid()
        if os.getpgrp() != self.pid or os.getsid(0) != self.pid:
            raise RuntimeError('fresh requires a dedicated kernel session with PGID == PID; refusing to signal shared group')
        self.out = Path(out); self.out.mkdir(parents=True, exist_ok=True)
        identity = Path(f'/proc/{self.pid}/stat').read_text().split(') ')[1].split()[19]
        now = time.monotonic()
        self.state = dict(started=now, overall=now+overall_seconds, inference=None, request=None, done=False)
        self._write()
        ready = self.out / 'watchdog-ready.json'
        ready.unlink(missing_ok=True)
        self.proc = subprocess.Popen([sys.executable, '-c', WATCHDOG_SOURCE, str(self.out), str(self.pid), identity], start_new_session=True)
        until = time.monotonic()+5
        while not ready.exists():
            if self.proc.poll() is not None or time.monotonic() > until:
                raise RuntimeError('independent watchdog failed startup')
            time.sleep(.01)
    def _write(self):
        p = self.out / 'watchdog-control.json'; t = p.with_suffix('.kernel.tmp')
        with t.open('w') as f:
            json.dump(self.state, f); f.flush(); os.fsync(f.fileno())
        t.replace(p)
    def inference(self, seconds=1200):
        self.state['inference'] = time.monotonic()+seconds; self._write()
    def request(self, seconds=120, metadata=None):
        self.state['request'] = time.monotonic()+seconds
        self.state['active_request'] = metadata; self._write()
    def clear_request(self):
        self.state['request'] = None; self.state['active_request'] = None; self._write()
    def inference_complete(self):
        self.state['inference'] = None; self._write()
    def abort(self, error):
        self.state['abort'] = str(error); self._write()
    def finish(self):
        self.state['done'] = True; self._write()
        self.proc.wait(timeout=5)
        assert self.proc.returncode == 0
        assert json.loads((self.out/'watchdog.json').read_text())['status'] == 'complete'


In [7]:
PRIVATE_LEDGER = Path.home() / '.local/state/llm-cost-frontier/drcd-openai-budget.json'

def public_snapshot(ledger, run_id, before):
    entries = {k: v for k, v in ledger.data['entries'].items() if v.get('run_id') == run_id}
    return {'run_id': run_id, 'auth_scope': AUTH_SCOPE, 'effective_cap_nano_usd': CAP_NANO,
        'authorized_nano_usd': 1_000_000_000, 'all_runs_exposure_nano_usd': ledger.exposure(),
        'all_runs_settled_nano_usd': sum(e.get('actual_nano_usd', 0) for e in ledger.data['entries'].values()),
        'all_runs_unknown_reserve_nano_usd': sum(e['reserve_nano_usd'] for e in ledger.data['entries'].values() if e['state'] == 'pending'),
        'metadata_request_count': ledger.data['metadata_request_count'] - before[0],
        'inference_request_count': ledger.data['inference_request_count'] - before[1],
        'all_runs_metadata_request_count': ledger.data['metadata_request_count'],
        'all_runs_inference_request_count': ledger.data['inference_request_count'],
        'halted': ledger.data['halted'], 'entries': entries, 'billing_reconciled': False}

def fresh_results(out, manifest_path, guard):
    authorize_fresh('fresh')
    manifest, manifest_bytes = load_manifest(manifest_path)
    out = Path(out)
    if (out / 'raw-generations.jsonl').exists(): raise APIStop('existing_raw_refuse_mix')
    # No alternate ledger path, cap override, reset switch, or provider URL option.
    with BudgetLedger(PRIVATE_LEDGER) as ledger:
        ledger.ensure_active()
        sender = make_transport('fresh')
        run_id = uuid.uuid4().hex
        before = (ledger.data['metadata_request_count'], ledger.data['inference_request_count'])
        records, preflight, decision = [], [], None
        status, stop_code = 'partial', None
        (out / 'sample-manifest.json').write_bytes(manifest_bytes)
        with (out / 'raw-generations.jsonl').open('xb') as raw_file:
            raw_file.flush(); os.fsync(raw_file.fileno())
        atomic_json(out / 'protocol.json', {'models': MODELS, 'model_ids_are_aliases': True,
            'manifest_sha256': MANIFEST_SHA, 'system': SYSTEM, 'max_output_tokens': MAX_OUTPUT,
            'reasoning': 'none', 'grid': GRID, 'retries': 0, 'planned_paid_calls': 90,
            'effective_cap_nano_usd': CAP_NANO, 'price_snapshot_date': '2026-11-10',
            'nano_usd_per_token': RATES, 'cache': 'explicit mode with no breakpoints',
            'short_input_limit': 8192, 'input_safety_tokens': 256, 'reserve_multiplier': 2,
            'usage_derived_not_billing_reconciled': True, 'run_id': run_id})
        def snapshot():
            value = public_snapshot(ledger, run_id, before)
            atomic_json(out / 'budget-snapshot.json', value)
            return value
        def guarded_sender(endpoint, payload):
            guard.request(120, {'endpoint': endpoint, 'model': payload['model']})
            try: return sender(endpoint, payload)
            finally: guard.clear_request()
        def on_reserved(token, record):
            snapshot()
            atomic_json(out / 'inflight.json', record)
        def dispatch(task):
            if len(records) >= 90: raise APIStop('request_count_limit')
            row = by_id[task['id']]
            payload = payload_for(row, task['model'])
            count = ledger.data['counts'][sha(canonical({'model': task['model'], 'input': payload['input']}))]
            metadata = {**task, 'order': len(records), 'run_id': run_id}
            failure = None
            try:
                result = paid_request(ledger, guarded_sender, payload, count, metadata, on_reserved=on_reserved)
            except APIStop as exc:
                if not hasattr(exc, 'record'): raise
                result, failure = exc.record, exc
            result['em'], result['f1'] = score(result['answer'], row['references'], result['error'])
            with (out / 'raw-generations.jsonl').open('ab') as stream:
                stream.write(canonical(result) + b'\n'); stream.flush(); os.fsync(stream.fileno())
            records.append(result)
            snapshot()
            (out / 'inflight.json').unlink(missing_ok=True)
            atomic_json(out / 'progress.json', {'run_id': run_id, 'collected': len(records), 'planned': 90})
            if failure is not None: raise failure
        by_id = {r['id']: r for k in ('warmup', 'dev', 'test') for r in manifest[k]}
        try:
            snapshot()
            # Metadata-only exact counts, all 33 unchanged inputs x 2 models, before paid inference.
            # Persistent cache keys include BOTH model and exact input structure.
            for split in ('warmup', 'dev', 'test'):
                for row in manifest[split]:
                    for model in MODELS:
                        payload = payload_for(row, model)
                        n = input_count(ledger, guarded_sender, payload)
                        preflight.append({'id': row['id'], 'model': model, 'input_sha256': sha(canonical(payload['input'])),
                            'exact_input_tokens': n, 'reserved_input_upper': n + 256,
                            'reserve_nano_usd': reservation(model, n)})
                        atomic_json(out / 'preflight.json', preflight)
                        snapshot()
            for task in initial_plan(manifest): dispatch(task)
            decision = select_decision(manifest, records)
            atomic_json(out / 'decision.json', decision)
            decision_bytes = (out / 'decision.json').read_bytes()
            atomic_json(out / 'decision-sha256.json', {'sha256': sha(decision_bytes), 'first_test_order': len(records)})
            # No test response is consulted by the dev selector; decision and SHA already durable.
            for task in test_plan(manifest, decision): dispatch(task)
            audit_records(manifest, records, decision)
            status = 'complete'
        except BaseException as exc:
            # Fixed codes only; never stringify HTTP exceptions or include provider messages.
            stop_code = str(exc) if isinstance(exc, APIStop) and str(exc) in {
                'budget_exhausted', 'counter_failed', 'inference_failed_or_ambiguous'} else 'local_or_protocol_failure'
            ledger.halt(stop_code)
        finally:
            budget = snapshot()
            report = summarize(records, status)
            report.update(stop_code=stop_code, decision=decision, budget_snapshot=budget,
                metadata_request_count=ledger.data['metadata_request_count'] - before[0],
                run_id=run_id, raw_sha256=sha((out / 'raw-generations.jsonl').read_bytes()))
            atomic_json(out / 'summary.json', report)
        return report


In [8]:
MODE = os.environ.get('DRCD_API_MODE', 'replay')
if MODE not in ('replay', 'fresh'): raise APIStop('invalid_mode')
ROOT = Path.cwd()
OUT = ROOT / 'results/drcd-api'
MANIFEST = ROOT / 'results/drcd-cpu/sample-manifest.json'
GUARD = None
if MODE == 'fresh':
    authorize_fresh(MODE)
    # An existing output directory is never overwritten or merged, even after a partial run.
    OUT.mkdir(parents=True, exist_ok=False)
    GUARD = ResourceGuard(OUT, overall_seconds=1200)
    def api_post_cell(result):
        if result.error_in_exec is not None or result.error_before_exec is not None:
            GUARD.abort('notebook_cell_failure')
        elif globals().get('guard_ready_to_finish', False):
            GUARD.finish()
            get_ipython().events.unregister('post_run_cell', api_post_cell)
    get_ipython().events.register('post_run_cell', api_post_cell)
    REPORT = fresh_results(OUT, MANIFEST, GUARD)
else:
    REPORT = replay_results(OUT, MANIFEST)
print(json.dumps({k: REPORT[k] for k in ('status', 'inference_request_count',
    'usage_derived_nano_usd', 'unknown_reserve_nano_usd', 'cost_basis')}, ensure_ascii=False, indent=2))


{
  "status": "complete",
  "inference_request_count": 90,
  "usage_derived_nano_usd": 19883700,
  "unknown_reserve_nano_usd": 0,
  "cost_basis": "usage-derived standard input/output; NOT billing reconciled"
}


In [9]:
# Plot saved, audited values only; no calls/downloads/credentials in this cell.
if REPORT['status'] == 'complete':
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    plt.rcParams.update({'figure.facecolor': '#1a1a2e', 'axes.facecolor': '#1a1a2e',
        'text.color': '#e0e0e0', 'axes.labelcolor': '#e0e0e0', 'xtick.color': '#e0e0e0',
        'ytick.color': '#e0e0e0', 'axes.edgecolor': '#555'})
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    metrics = [REPORT['test'][scheme] for scheme in SCHEMES]
    axes[0].bar(SCHEMES, [m['usage_derived_nano_usd'] / 1e9 for m in metrics], color=['#9b8cff', '#5dd4c4', '#ffbd69'])
    axes[0].set_ylabel('Usage-derived USD (not reconciled bill)')
    for key, color in [('em', '#9b8cff'), ('f1', '#5dd4c4')]:
        axes[1].plot(SCHEMES, [m[key] for m in metrics], 'o-', label=key, color=color)
    axes[1].set_ylim(0, 1); axes[1].legend()
    fig.suptitle('DRCD official API: ' + REPORT['decision']['label'])
    fig.text(.5, .01, '24 unchanged public historical questions per scheme; dev-only frozen routes; standard prices 2026-11-10', ha='center', color='#aaa', fontsize=9)
    fig.tight_layout(rect=(0, .04, 1, .95))
    fig.savefig(OUT / ('comparison.png' if MODE == 'fresh' else 'comparison-replayed.png'), dpi=150, facecolor='#1a1a2e')
    plt.close(fig)
else:
    print('No complete paid evaluation: no cost/quality frontier claimed, no synthetic plot.')


In [10]:
if MODE == 'fresh':
    # Re-read public evidence while the overall watchdog is still armed.
    AUDIT = replay_results(OUT, MANIFEST)
    assert AUDIT['status'] == REPORT['status']
    assert AUDIT['budget_snapshot']['all_runs_exposure_nano_usd'] <= CAP_NANO
else:
    assert REPORT['status'] in ('not_run', 'partial', 'complete')
print('Offline evidence audit complete; replay does not imply fresh inference or billing reconciliation.')
if MODE == 'fresh':
    guard_ready_to_finish = True  # disarm in post_run_cell only after this body succeeds


Offline evidence audit complete; replay does not imply fresh inference or billing reconciliation.
